# התחלת עבודה עם נתונים
Import pandas and load the CSV file.

In [1]:
import pandas as pd

url = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"

try:
    df = pd.read_csv(url,
                     compression='zip',
                     nrows=10000,
                     low_memory=False)

    display(df.head())

except Exception as e:
    print(f"Error loading data: {e}")
# הצגת כל העמודות בלי חיתוך
pd.set_option('display.max_columns', None)

# אופציונלי – שלא יקצר גם רוחב תצוגה
pd.set_option('display.width', 2000)

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,2020-07-06,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,346XX,NaN,Other,Web,2020-07-06,Closed with explanation,Yes,NaN,3730948
1,2019-12-26,Credit card or prepaid card,General-purpose credit card or charge card,"Advertising and marketing, including promotion...",Confusing or misleading advertising about the ...,NaN,NaN,CAPITAL ONE FINANCIAL CORPORATION,CA,94025,NaN,Consent not provided,Web,2019-12-26,Closed with explanation,Yes,NaN,3477549
2,2020-05-08,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,These are not my accounts.,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,NV,89030,NaN,Consent provided,Web,2020-05-08,Closed with explanation,Yes,NaN,3642453
3,2024-01-05,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,Kindly address this issue on my credit report....,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,IL,60502,NaN,Consent provided,Web,2024-01-05,Closed with non-monetary relief,Yes,NaN,8113747
4,2024-01-21,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Credit inquiries on your report that you don't...,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,NC,27401,Servicemember,Consent not provided,Web,2024-01-21,Closed with explanation,Yes,NaN,8191825


In [2]:
# הצגת 6 שורות ראשונות
print(df.head(6).to_string())

  Date received                                                                       Product                                 Sub-product                                                    Issue                                                  Sub-issue                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [3]:
# 1. הסרת שורות שאין בהן את תיאור התלונה (חובה לחיזוי)
if 'Consumer complaint narrative' in df.columns:
    df = df.dropna(subset=['Consumer complaint narrative'])

# 2. רשימת העמודות שאנחנו רוצים לנסות למחוק
columns_to_drop = ['Tags', 'Consumer consent provided?', 'Complaint ID', 'ZIP code']

# 3. מחיקה בטוחה - גם אם העמודה לא קיימת, לא תהיה שגיאה
df = df.drop(columns=columns_to_drop, errors='ignore')

print(f"הניקוי הצליח! נשארו לנו {len(df)} שורות לעבודה.")
df.head()

הניקוי הצליח! נשארו לנו 2663 שורות לעבודה.


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?
2,2020-05-08,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,These are not my accounts.,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,NV,Web,2020-05-08,Closed with explanation,Yes,NaN
3,2024-01-05,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,Kindly address this issue on my credit report....,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,IL,Web,2024-01-05,Closed with non-monetary relief,Yes,NaN
5,2020-03-19,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,"I wrote three requests, the unverified account...",NaN,"EQUIFAX, INC.",NC,Web,2020-03-19,Closed with explanation,Yes,NaN
6,2019-10-22,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Old information reappears or never goes away,XXXX XXXX has a old account settled in XXXX th...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,HI,Web,2019-10-22,Closed with non-monetary relief,Yes,NaN
7,2020-03-29,Student loan,Federal student loan servicing,Dealing with your lender or servicer,Keep getting calls about your loan,They call at all hours and on the weekends usi...,NaN,"Navient Solutions, LLC.",CA,Web,2020-03-29,Closed with explanation,Yes,NaN


In [4]:
# פונקציה שעושה סדר בשמות המוצרים
def unite_products(product):
    product = str(product)
    if 'Credit reporting' in product or 'Credit repair' in product:
        return 'Credit Services'
    if 'Mortgage' in product:
        return 'Mortgage'
    if 'Loan' in product:
        return 'Loans'
    if 'Bank' in product or 'Checking' in product or 'Savings' in product:
        return 'Banking Services'
    if 'Debt collection' in product:
        return 'Debt Collection'
    return 'Other'

# יצירת עמודה חדשה ומסודרת
df['Product_Target'] = df['Product'].apply(unite_products)

# בדיקה כמה יש לנו מכל סוג עכשיו
print(df['Product_Target'].value_counts())

Product_Target
Credit Services     1754
Other                372
Debt Collection      314
Banking Services     120
Mortgage              94
Loans                  9
Name: count, dtype: int64


In [5]:
df.info()
df['Product'].value_counts()
df['Sub-product'].value_counts()

<class 'pandas.core.frame.DataFrame'>
Index: 2663 entries, 2 to 9990
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   Date received                 2663 non-null   object
 1   Product                       2663 non-null   object
 2   Sub-product                   2628 non-null   object
 3   Issue                         2663 non-null   object
 4   Sub-issue                     2435 non-null   object
 5   Consumer complaint narrative  2663 non-null   object
 6   Company public response       1448 non-null   object
 7   Company                       2663 non-null   object
 8   State                         2650 non-null   object
 9   Submitted via                 2663 non-null   object
 10  Date sent to company          2663 non-null   object
 11  Company response to consumer  2663 non-null   object
 12  Timely response?              2663 non-null   object
 13  Consumer disputed?     

,count
Sub-product,
Credit reporting,1718
General-purpose credit card or charge card,123
I do not know,95
Checking account,88
Other debt,64
Credit card debt,61
Conventional home mortgage,56
Domestic (US) money transfer,51
Loan,30


כאן אנחנו לומדם על הנתונים שלנו

In [6]:
word_counts = df['Consumer complaint narrative'].str.split().str.len()


# 2. חישוב הממוצע לפי נושא ושמירה במשתנה רגיל
avg_per_topic = word_counts.groupby(df['Product']).mean().sort_values(ascending=False)

# 3. הדפסת התוצאות
print("ממוצע מילים לכל נושא (במשתנה avg_per_topic):")
print(avg_per_topic)

ממוצע מילים לכל נושא (במשתנה avg_per_topic):
Product
Mortgage                                                                        297.946809
Debt or credit management                                                       265.166667
Credit card                                                                     264.600000
Checking or savings account                                                     249.616071
Vehicle loan or lease                                                           230.833333
Credit card or prepaid card                                                     226.521739
Student loan                                                                    217.785714
Bank account or service                                                         215.375000
Payday loan, title loan, or personal loan                                       207.700000
Money transfer, virtual currency, or money service                              182.402299
Debt collection                      

In [7]:
# 1. בדיקת איזון - כמה תלונות יש לכל נושא
class_counts = df['Product'].value_counts()

# 2. בדיקת כפילויות - כמה שורות הן העתק מדויק
duplicate_count = df.duplicated().sum()

print("התפלגות הנושאים:")
print(class_counts)
print(f"\nמספר שורות כפולות במאגר: {duplicate_count}")

התפלגות הנושאים:
Product
Credit reporting or other personal consumer reports                             1090
Credit reporting, credit repair services, or other personal consumer reports     645
Debt collection                                                                  314
Checking or savings account                                                      112
Mortgage                                                                          94
Credit card or prepaid card                                                       92
Money transfer, virtual currency, or money service                                87
Credit card                                                                       80
Student loan                                                                      42
Vehicle loan or lease                                                             36
Credit reporting                                                                  19
Payday loan, title loan, or personal loa

כמה גרפים מעניניים


In [ ]:
import matplotlib.pyplot as plt
# גרף המדינות המובילות בתלונות
df['State'].value_counts().head(10).plot(kind='pie', autopct='%1.1f%%', figsize=(8,8))
plt.title('10 המדינות עם הכי הרבה תלונות')
plt.ylabel('')
plt.show()

In [ ]:
# חישוב אורך כל תלונה במילים
df['complaint_length'] = df['Consumer complaint narrative'].str.split().str.len()

# ציור גרף שמראה את התפלגות האורך
df['complaint_length'].plot(kind='hist', bins=50, color='lightgreen', figsize=(10,6))
plt.title('התפלגות אורך התלונות (במילים)')
plt.xlabel('מספר מילים')
plt.ylabel('כמות תלונות')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. ניקח את 10 הנושאים הכי נפוצים כדי שהגרף יהיה קריא
top_issues = df['Issue'].value_counts().nlargest(10).index
df_subset = df[df['Issue'].isin(top_issues)]

# 2. יצירת טבלת הצלבה (Cross-tabulation)
ct = pd.crosstab(df_subset['Issue'], df_subset['Product'])

# 3. יצירת גרף מפת חום (Heatmap)
plt.figure(figsize=(12, 8))
sns.heatmap(ct, annot=True, fmt="d", cmap="YlGnBu")
plt.title("קשר בין נושא (Issue) למוצר (Product)")
plt.xlabel("מוצר")
plt.ylabel("נושא")
plt.show()

In [ ]:

df = df[df["Consumer complaint narrative"].notna() & (df["Consumer complaint narrative"].str.strip() != "")]
plt.figure(figsize=(12, 6))

# חישוב אורך טקסט בלי להוסיף עמודה ל־DF
text_lengths = df['Consumer complaint narrative'] \
    .fillna('') \
    .str.split() \
    .str.len()

# סינון עד 1000 מילים
filtered_lengths = text_lengths[text_lengths <= 1000]

# גרף
sns.histplot(
    filtered_lengths,
    bins=50,
    kde=True,
    color='teal'
)

# ממוצע וחציון
plt.axvline(
    text_lengths.mean(),
    color='orange',
    linestyle='--',
    label=f'Mean: {int(text_lengths.mean())}'
)

plt.axvline(
    text_lengths.median(),
    color='red',
    linestyle='-',
    label=f'Median: {int(text_lengths.median())}'
)

plt.title('Focus: Word Count Distribution (0-1000 words)', fontsize=15)
plt.xlabel('Number of Words')
plt.ylabel('Number of Complaints')
plt.legend()
plt.grid(axis='y', alpha=0.3)

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Filtering top 10 states
top_states = df['State'].value_counts().nlargest(10).index
df_top_states = df[df['State'].isin(top_states)]

# 2. Creating normalized cross-tabulation
state_product_pct = pd.crosstab(df_top_states['State'], df_top_states['Product'], normalize='index') * 100

# 3. Plotting with brighter colors (Set3 or Pastel1)
plt.figure(figsize=(15, 8))
state_product_pct.plot(kind='bar',
                       stacked=True,
                       colormap='Accent', # Brighter, distinct colors
                       ax=plt.gca(),
                       edgecolor='white',
                       linewidth=0.5)

# Styling the plot in English
plt.title("Product Distribution by State (Top 10 States)", fontsize=16, pad=20)
plt.xlabel("State Code", fontsize=12)
plt.ylabel("Percentage of Complaints (%)", fontsize=12)
plt.legend(title="Product Categories", bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
plt.xticks(rotation=0) # Keeps state names horizontal for clarity
plt.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

סיכום מקצועי (Project Summary)
דמיון גיאוגרפי: הניתוח מראה כי סוגי התלונות אינם משתנים באופן מובהק בין המדינות השונות בארה"ב. המשמעות היא שמיקום הלקוח אינו מהווה אינדיקציה טובה לחיזוי סוג התלונה.

חוסר איזון במחלקות (Class Imbalance): קטגוריית ה-Credit Reporting שולטת בנתונים בצורה אבסולוטית. ללא טיפול באיזון הנתונים (כמו Oversampling), המודל עלול "להתעצל" ולסווג כמעט הכל כקטגוריה זו.

התמקדות בטקסט: לאור הממצאים שהפיצ'רים היבשים (כמו מדינה) הם חלשים, המודל חייב להישען על עיבוד שפה טבעית (NLP) כדי להצליח בסיווג.

In [ ]:
cols_to_drop = [
    'Tags', 'Company public response', 'Complaint ID', 'ZIP code',
    'Date received', 'Date sent to company', 'Consumer consent provided?', 'Sub-issue',
    'Submitted via', 'Timely response?', 'Company response to consumer', 'Consumer disputed?'
]

# הסרה מה-DataFrame
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

print("Final features for model training:")
print(df.columns)

בשלב ניקוי הנתונים (Data Cleaning), הוסרו עמודות המכילות מידע חסר באופן גורף (כמו Consumer disputed? ו-Tags), שכן הן אינן תורמות "רווח מידע" (Information Gain) למודל. בנוסף, הסרנו מזהים ייחודיים כמו Complaint ID ונתונים מנהלתיים כגון Date received ו-Submitted via, אשר מהווים "רעש" סטטיסטי ועלולים לגרום למודל לפתח התאמת יתר (Overfitting) במקום ללמוד דפוסים אמיתיים בטקסט התלונה. מטרת ההסרה היא למקד את מודל ה-Logistic Regression בפיצ'רים בעלי עוצמה חיזויית גבוהה—בראשם תוכן התלונה (Consumer complaint narrative) והנושא (Issue)—כדי להבטיח סיווג מדויק ויעיל של המוצרים הפיננסיים

In [ ]:
# mapping to general categories
mapping = {
    "Credit reporting or other personal consumer reports": "Credit",
    "Credit card": "Credit",

    "Debt collection": "Debt",
    "Debt or credit management": "Debt",

    "Checking or savings account": "Banking",
    "Prepaid card": "Banking",

    "Money transfer, virtual currency, or money service": "Payments",

    "Mortgage": "Loans",
    "Vehicle loan or lease": "Loans",
    "Student loan": "Loans",
    "Payday loan, title loan, personal loan, or advance loan": "Loans",
}

# create new column with default category
df["category"] = df["Product"].map(mapping).fillna("Other")


# check which values were not mapped
print("Unmapped values:")
print(df[df["category"] == "Other"]["Product"].unique())


# check distribution
print("\nCategory distribution:")
print(df["category"].value_counts())


# optional plot
import matplotlib.pyplot as plt

df["category"].value_counts().plot(kind="bar", figsize=(8,5), title="Category Distribution")
plt.show()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
import seaborn as sns
import matplotlib.pyplot as plt

# 1. יצירת רשימת מילים לסינון - הוספנו את ה"איקסים" לרשימה הקיימת באנגלית
my_stop_words = list(ENGLISH_STOP_WORDS) + ['xxxx', 'xx', 'xx/xx/xxxx', '00']

# 2. פונקציה מעודכנת למציאת צמדי מילים בלי האיקסים
def get_top_n_bigrams_clean(corpus, n=None):
    vec = CountVectorizer(ngram_range=(2, 2), stop_words=my_stop_words).fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key = lambda x: x[1], reverse=True)
    return words_freq[:n]

# 3. הרצה על הנתונים שלנו
print("מנתח את הטקסט (זה עשוי לקחת כדקה)...")
top_bigrams = get_top_n_bigrams_clean(df['Consumer complaint narrative'], n=10)
x, y = map(list, zip(*top_bigrams))

# 4. ציור הגרף הנקי
plt.figure(figsize=(12,6))
sns.barplot(x=y, y=x, palette='magma')
plt.title('10 צמדי המילים הנפוצים ביותר (אחרי סינון איקסים)')
plt.xlabel('מספר מופעים')
plt.ylabel('צמד מילים')
plt.show()

מה עשינו: זיהוי הקטגוריה עם מספר המופעים הנמוך ביותר ודגימה אקראית של כמות שורות זהה מכל שאר הקטגוריות.

למה: זהו השלב הקריטי למניעת הטיה (Bias). מודל שמתאמן על נתונים לא מאוזנים נוטה "לנחש" תמיד את הקטגוריה הגדולה ביותר. איזון הנתונים מבטיח שלכל קטגוריה יהיה משקל שווה בתהליך הלמידה.

In [ ]:
df["category"].value_counts().plot(kind="bar", figsize=(8,5), title="Category Distribution")


In [ ]:
import pandas as pd

url = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"
chunk_size = 5000
chunks = []

# 1. טעינת הנתונים ב-Chunks וביצוע עיבוד ראשוני
for chunk in pd.read_csv(url, chunksize=chunk_size):
    # הסרת עמודות מיותרות
    chunk.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    # מיפוי קטגוריות
    chunk["category"] = chunk["Product"].map(mapping).fillna("Other")

    # סינון קטגוריית Credit
    chunk_filtered = chunk[chunk["category"] != "Credit"].copy()

    chunks.append(chunk_filtered)

# 2. איחוד כל החלקים
df_combined_full = pd.concat(chunks, ignore_index=True)

# 3. איחוד עם DF קיים אם יש
if 'df' in locals():
    df["category"] = df["Product"].map(mapping).fillna("Other")
    df_combined_full = pd.concat([df, df_combined_full], ignore_index=True)

print("Columns in combined DataFrame:", df_combined_full.columns.tolist())

# 4. חישוב גודל חציון לכל קטגוריה
category_counts = df_combined_full["category"].value_counts()
median_size = int(category_counts.median())

print(f"\nMedian category size: {median_size}")

# 5. איזון הנתונים לפי החציון (Oversampling לקטנות בלבד)
balanced_chunks = []

for category, group in df_combined_full.groupby("category"):
    if len(group) < median_size:
        # שכפול עם החזרה (oversampling)
        resampled = group.sample(n=median_size, replace=True, random_state=42)
    else:
        # השארת הדאטה כמו שהוא (לא חותכים)
        resampled = group.copy()

    balanced_chunks.append(resampled)

df_balanced = pd.concat(balanced_chunks, ignore_index=True)

# 6. תוצאות
print("\nCounts per category in balanced DataFrame:")
print(df_balanced["category"].value_counts())

print(f"\nTotal rows in balanced DF: {len(df_balanced)}")

In [ ]:
df_balanced = df_balanced[df_balanced["Consumer complaint narrative"].notna() & (df_balanced["Consumer complaint narrative"].str.strip() != "")]
print(df_balanced["category"].value_counts())

print(f"\nTotal rows: {len(df_balanced)}")


In [ ]:
df=df_balanced


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
path = "/content/drive/MyDrive/df.csv"

df.to_csv(path, index=False)

print("Saved successfully!")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# נתונים
X = df["Consumer complaint narrative"].astype(str)
y = df["category"]

# split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

fast_model = Pipeline([
    (
        "vectorizer",
        HashingVectorizer(
            lowercase=True,
            stop_words="english",

            # מהיר מאוד
            n_features=2**18,

            # unigram בלבד
            ngram_range=(1, 1),

            alternate_sign=False
        )
    ),
    (
        "clf",
        SGDClassifier(
            loss="hinge",          # SVM
            penalty="l2",

            # מהיר
            max_iter=10,

            # עצירה מוקדמת
            early_stopping=True,

            # שימוש בכל הליבות
            n_jobs=-1,

            random_state=42
        )
    )
])

# אימון
fast_model.fit(X_train, y_train)

# חיזוי
preds = fast_model.predict(X_test)

# תוצאות
print("Accuracy:", accuracy_score(y_test, preds))

print(classification_report(y_test, preds))

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

# 1. השתמשנו ב-preds כי זה השם שמופיע בקוד המהיר שלך
print(f"Accuracy Score: {accuracy_score(y_test, preds):.2f}")

# 2. דוח ביצועים
print("\nClassification Report:")
print(classification_report(y_test, preds))

# 3. מטריצת בלבול
cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(12,8))

# משתמשים ב-fast_model.classes_ כדי להוציא את שמות הקטגוריות
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=fast_model.classes_,
            yticklabels=fast_model.classes_)

plt.xlabel('Predicted (מה שהמודל ניחש)')
plt.ylabel('Actual (התשובה האמיתית)')
plt.title('Confusion Matrix - Fast SGD Model')
plt.show()

 סיכום תוצאות ומסקנות המחקר 📊
📈 ניתוח ביצועי המודל

בפרויקט זה בוצע סיווג אוטומטי של פניות צרכנים באמצעות מודל SGDClassifier (במימוש של Linear SVM).
המודל הציג ביצועים טובים יחסית למורכבות הנתונים, עם Accuracy של כ-74%.

עם זאת, ניתוח מעמיק של המדדים מראה כי הדיוק הכולל לבדו אינו מספיק כדי להעריך את איכות המודל. למרות שבחלק מהקטגוריות כמו Payments ו-Loans התקבלו ערכי F1-Score גבוהים (מעל 0.80), בקטגוריות אחרות התקבלו ערכים נמוכים משמעותית — דבר המעיד על קושי של המודל להבחין בין מחלקות בעלות שפה דומה.

🔍 תובנות מרכזיות מהניתוח
זיהוי מבוסס מילים (Feature Importance)

המודל מתבסס בעיקר על הופעת מילים בודדות ותדירותן בטקסט.
בקטגוריות שבהן קיימים מונחים מקצועיים ברורים כמו:

“Loan”
“Mortgage”
“Transfer”

המודל מצליח לזהות את הקטגוריה בצורה טובה מאוד.

מגבלת ההבנה הסמנטית

אחת התובנות המרכזיות מהמחקר היא שהמודל אינו מבין את המשמעות המלאה של המשפט, אלא רק דפוסים סטטיסטיים של מילים.

לכן, כאשר קיימת חפיפה סמנטית בין קטגוריות — למשל בין:

Credit
Banking

המודל מתקשה להבדיל ביניהן.

מילים כמו:

“account”
“bank”
“card”

מופיעות במספר קטגוריות שונות, אך המשמעות שלהן משתנה לפי ההקשר.
מאחר שהמודל אינו מבין הקשר תחבירי או משמעות סמנטית עמוקה, הוא מבצע לעיתים סיווג שגוי גם כאשר המשפט ברור לבן אדם.

אתגר קטגוריית "Other"

קטגוריית Other הציגה את ביצועי ה-F1 הנמוכים ביותר.
הסיבה לכך היא שמדובר בקטגוריה כללית הכוללת מגוון רחב של נושאים ללא מאפיינים טקסטואליים אחידים. כתוצאה מכך, למודל קשה ללמוד דפוס זיהוי יציב וברור.

💡 מסקנות סופיות והמלצות
חשיבות עיבוד הנתונים

שלב ניקוי הנתונים היה קריטי להצלחת המודל:

הסרת רצפים כמו XXXX
סינון Stop Words
ניקוי טקסטים לא רלוונטיים

ללא שלבים אלו, המודל היה לומד רעש במקום מידע משמעותי.

יעילות חישובית

השילוב של HashingVectorizer יחד עם SGDClassifier אפשר עיבוד מהיר ויעיל של כמות גדולה של נתונים, תוך שימוש נמוך בזיכרון וזמני אימון קצרים.

מגבלת המודל הקלאסי

למרות היעילות והמהירות, תוצאות המחקר מראות כי מודל מבוסס Linear SVM אינו מספיק חזק עבור משימת סיווג טקסט מורכבת, במיוחד כאשר קיימות מחלקות עם חפיפה סמנטית גבוהה וערכי F1 נמוכים.

המודל מצליח לזהות מילים — אך לא להבין את המשמעות הכוללת של המשפט.

כיוון להמשך פיתוח – מעבר למודל LLM

בעקבות הממצאים, הוחלט לאמן מודל מבוסס LLM (Large Language Model), כגון BERT, אשר מסוגל להבין:

הקשר בין מילים
משמעות סמנטית
מבנה משפט
כוונת המשתמש

מודלים אלו אינם מסתמכים רק על מילים בודדות, אלא לומדים את משמעות המשפט כולו.
לכן, הם צפויים לשפר משמעותית את ביצועי הסיווג במחלקות שבהן התקבלו ערכי F1 נמוכים ולהפחית את הבלבול בין קטגוריות דומות

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/df.csv"

df = pd.read_csv(
    path,
    usecols=["Consumer complaint narrative", "category"]
)

print(df.shape)

In [ ]:
df = df.drop_duplicates()

# למצוא את גודל הקטגוריה הקטנה ביותר
min_size = df["category"].value_counts().min()

print("Smallest category size:", min_size)

# איזון לפי הקטגוריה הקטנה ביותר
df_balanced = (
    df.groupby("category", group_keys=False)
      .sample(n=min_size, random_state=42)
)

# ערבוב הנתונים
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# תוצאות
print(df_balanced["category"].value_counts())

print(df_balanced.shape)

print(df_balanced.head())

In [ ]:
import gc

del df
gc.collect()

In [ ]:


df=df_balanced

In [ ]:
#חלוקת דאטה
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["category"],
    random_state=42
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["category"],
    random_state=42
)

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(train_df["category"])

valid_df["label"] = label_encoder.transform(valid_df["category"])

test_df["label"] = label_encoder.transform(test_df["category"])

In [ ]:
for i, label in enumerate(label_encoder.classes_):
    print(i, "->", label)

In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
sample_text = train_df["Consumer complaint narrative"].iloc[0]

tokens = tokenizer(sample_text)

print(tokens.keys())


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["Consumer complaint narrative"],
        truncation=True,

        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)

valid_dataset = valid_dataset.map(tokenize_function, batched=True)

test_dataset = test_dataset.map(tokenize_function, batched=True)

In [ ]:
from transformers import AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_encoder.classes_)
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)



In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/checkpoints",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    num_train_epochs=1,

    weight_decay=0.01,

    fp16=True,

    logging_steps=100,

    save_total_limit=1,

    load_best_model_at_end=False,

    report_to="none"
)

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=valid_dataset,

    data_collator=data_collator
)
trainer.train()


In [ ]:
import shutil
from google.colab import drive
drive.mount('/content/drive')

# תיקיית המודל
model_path = "/content/drive/MyDrive/checkpoints/final_model"

# שמירת tokenizer + model
tokenizer.save_pretrained(model_path)
trainer.save_model(model_path)

# יצירת zip
shutil.make_archive(
    "/content/drive/MyDrive/final_model",
    'zip',
    model_path
)

print("Model zipped successfully")